In [90]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import os
import ast

os.makedirs("../Figures/SHAP_Analysis", exist_ok=True)
os.makedirs("../Figures/SHAP_Analysis/Waterfalls_80", exist_ok=True)
os.makedirs("../Figures/SHAP_Analysis/Waterfalls_90", exist_ok=True)
os.makedirs("../data/Results/SHAP", exist_ok=True)

print("Loading model artifacts...")
artifacts = joblib.load("../data/Shap_Data/Chembl/model_artifacts_chembl.pkl")

print("Loading prediction results...")
results_df = pd.read_excel("../data/Shap_Data/Chembl/Chembl_Screening.xlsx")

print(f"Model has {len(artifacts['feature_names'])} features")
print(f"Total predictions: {len(results_df)}")

#convert fingerprints from string to array
fingerprints = []
for fp in results_df['Filtered_Fingerprint']:
    if isinstance(fp, str):
        fp_list = ast.literal_eval(fp)
        fingerprints.append(np.array(fp_list, dtype=int))
    else:
        fingerprints.append(np.array(fp, dtype=int))
X_all = np.vstack(fingerprints)

In [89]:
print("\nCalculating SHAP values for ALL predictions...")
explainer = shap.TreeExplainer(
    artifacts['model'],
    X_all,
    feature_perturbation='interventional',
    model_output='probability'
)

shap_values_all = explainer.shap_values(X_all)

#extract binders
if isinstance(shap_values_all, list):
    shap_values_all_class1 = shap_values_all[1]
elif len(shap_values_all.shape) == 3:
    shap_values_all_class1 = shap_values_all[:, :, 1]
else:
    shap_values_all_class1 = shap_values_all

if isinstance(explainer.expected_value, (list, np.ndarray)):
    expected_value = explainer.expected_value[1]
else:
    expected_value = explainer.expected_value

print(f"SHAP values shape (all data): {shap_values_all_class1.shape}")
print(f"Base probability (expected value): {expected_value:.4f}")

In [88]:
print("\nCalculating feature importance from ALL data...")
mean_abs_shap_all = np.abs(shap_values_all_class1).mean(axis=0)
sorted_indices = np.argsort(mean_abs_shap_all)[::-1]
sorted_importance = mean_abs_shap_all[sorted_indices]
cumulative_importance = np.cumsum(sorted_importance) / np.sum(sorted_importance)

#find 80% and 90% thresholds
threshold_80_idx = np.where(cumulative_importance >= 0.8)[0][0]
threshold_90_idx = np.where(cumulative_importance >= 0.9)[0][0]
num_features_80 = threshold_80_idx + 1
num_features_90 = threshold_90_idx + 1

print(f"\nFeatures for 80% cumulative importance: {num_features_80}")
print(f"Features for 90% cumulative importance: {num_features_90}")

#top features for each threshold, based on all the data
top_features_80 = [artifacts['feature_names'][i] for i in sorted_indices[:num_features_80]]
top_features_90 = [artifacts['feature_names'][i] for i in sorted_indices[:num_features_90]]

In [86]:
positive_mask = results_df['Probability'] > 0.5

positive_results = results_df[positive_mask].copy()
positive_indices = positive_mask[positive_mask].index.tolist()
shap_values_positive = shap_values_all_class1[positive_mask]
X_positive = X_all[positive_mask]

print(f"\nPositive predictions to display: {len(positive_results)}")

In [87]:
print(positive_results)

In [85]:
#Cumulative Importance Graph
print("\n[1/4] Creating cumulative importance plot (based on all data)...")

plt.figure(figsize=(12, 7))
plt.plot(range(1, len(cumulative_importance) + 1), cumulative_importance, 
         linewidth=2.5, color='darkblue')
plt.fill_between(range(1, len(cumulative_importance) + 1), cumulative_importance, 
                 alpha=0.3, color='lightblue')
plt.axhline(y=0.8, color='red', linestyle='--', linewidth=2, label='80% Threshold', alpha=0.7)
plt.axhline(y=0.9, color='orange', linestyle='--', linewidth=2, label='90% Threshold', alpha=0.7)

plt.axvline(x=num_features_80, color='red', linestyle=':', linewidth=2, alpha=0.6)
plt.axvline(x=num_features_90, color='orange', linestyle=':', linewidth=2, alpha=0.6)

plt.xlabel('Number of Features', fontsize=12, fontweight='bold')
plt.ylabel('Cumulative Importance', fontsize=12, fontweight='bold')
plt.title('Cumulative Feature Importance: False Negatives', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.xlim(1, len(cumulative_importance))
plt.ylim(0, 1)

plt.text(num_features_80, 0.85, f'{num_features_80} features\n(80.0%)', 
         ha='center', fontsize=10, 
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
plt.text(num_features_90, 0.95, f'{num_features_90} features\n(90.0%)', 
         ha='center', fontsize=10,
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig("../Figures/SHAP_Analysis/cumulative_importance.svg", dpi=300, bbox_inches='tight')
plt.show()

In [84]:
#Waterfall plots for individual positive predictions
print(f"\n[4/4] Creating waterfall plots for {len(positive_results)} positive predictions...")
print("(Using feature rankings from all data)")
print("This may take a few minutes...")

chemical_names = positive_results['Name'].tolist()
probabilities = positive_results['Probability'].tolist()

#for 80% threshold waterfall plots
print(f"\nGenerating {len(positive_results)} waterfall plots (top {num_features_80} features - 80% threshold)...")
for i in range(len(positive_results)):
    fig, ax = plt.subplots(figsize=(10, max(8, num_features_80 * 0.4)))
    
    explanation = shap.Explanation(
        values=shap_values_positive[i],
        base_values=expected_value,
        data=X_positive[i],
        feature_names=artifacts['feature_names']
    )
    
    shap.waterfall_plot(
        explanation,
        show=False,
        max_display=num_features_80 + 1
    )
    
    plt.title(f"ChEMBL Screening\n{chemical_names[i]}\nProbability={probabilities[i]:.4f} (Top {num_features_80} Features - 80% Threshold)", 
              fontsize=12, fontweight='bold', pad=15)
    
    plt.tight_layout()
    safe_name = "".join(c if c.isalnum() or c in (' ', '-', '_') else '_' for c in chemical_names[i])
    plt.savefig(f"../Figures/SHAP_Analysis/Waterfalls_80/{i+1:03d}_{safe_name[:50]}.svg", 
                dpi=300, bbox_inches='tight')
    plt.close()
    
    if (i + 1) % 10 == 0:
        print(f"  Completed {i + 1}/{len(positive_results)} (80% threshold)...")

print(f"\nCompleted all {len(positive_results)} waterfall plots (top {num_features_80} features)")

#for 90% threshold waterfall plots
print(f"\nGenerating {len(positive_results)} waterfall plots (top {num_features_90} features - 90% threshold)...")
for i in range(len(positive_results)):
    fig, ax = plt.subplots(figsize=(10, max(10, num_features_90 * 0.4)))
    
    explanation = shap.Explanation(
        values=shap_values_positive[i],
        base_values=expected_value,
        data=X_positive[i],
        feature_names=artifacts['feature_names']
    )
    
    shap.waterfall_plot(
        explanation,
        show=False,
        max_display=num_features_90 + 1
    )
    
    plt.title(f"{chemical_names[i]}\nProbability={probabilities[i]:.4f} (Top {num_features_90} Features - 90% Threshold)", 
              fontsize=12, fontweight='bold', pad=15)
    
    plt.tight_layout()
    safe_name = "".join(c if c.isalnum() or c in (' ', '-', '_') else '_' for c in chemical_names[i])
    plt.savefig(f"../Figures/SHAP_Analysis/Waterfalls_90/{i+1:03d}_{safe_name[:50]}.svg", 
                dpi=300, bbox_inches='tight')
    plt.close()
    
    if (i + 1) % 10 == 0:
        print(f"  Completed {i + 1}/{len(positive_results)} (90% threshold)...")

In [82]:
#Summary
print("\n" + "="*70)
print("FOCUSED SHAP ANALYSIS COMPLETE")
print("="*70)
print(f"\nTotal predictions analyzed: {len(results_df)}")
print(f"Positive predictions displayed: {len(positive_results)}")
print(f"\nTop {num_features_80} Features (80% importance threshold - from all data):")
for i, feat in enumerate(top_features_80, 1):
    feat_idx = artifacts['feature_names'].index(feat)
    print(f"  {i:2d}. {feat:15s} - Mean |SHAP|: {mean_abs_shap_all[feat_idx]:.4f}")

print(f"\nTop {num_features_90} Features (90% importance threshold - from all data):")
for i, feat in enumerate(top_features_90, 1):
    feat_idx = artifacts['feature_names'].index(feat)
    print(f"  {i:2d}. {feat:15s} - Mean |SHAP|: {mean_abs_shap_all[feat_idx]:.4f}")

print("\n\nGenerated Files:")
print("  Main Figures:")
print("    - cumulative_importance.svg")
print(f"    - shap_correlation_80pct.svg (top {num_features_80} features)")
print(f"    - shap_correlation_90pct.svg (top {num_features_90} features)")
print(f"\n  Waterfall Plots (Positive Predictions Only):")
print(f"    - Waterfalls_80/ ({len(positive_results)} plots with top {num_features_80} features)")
print(f"    - Waterfalls_90/ ({len(positive_results)} plots with top {num_features_90} features)")
print("\n  Data:")
print("    - feature_importance_ranked.xlsx")
print("\n" + "="*70)
print("\nNote: All statistics and feature rankings are calculated from ALL predictions.")
print("The 80% and 90% thresholds are determined by cumulative importance.")
print("Waterfall plots only show positive predictions.")
print("="*70)

In [83]:
#THIS WAS A FOR A TEST WE DID FOR FALSE NEGATIVES - NOT PART OF THE MAIN ANALYSIS, BUT LEFT IN FOR REFERENCE
#Decision plot false neagtives
if results_df['Label'].dtype == 'object':
    label_mask = results_df['Label'].astype(str) == '0'
else:
    label_mask = results_df['Label'] == 0

if results_df['Prediction'].dtype == 'object':
    pred_mask = results_df['Prediction'].astype(str) == '1'
else:
    pred_mask = results_df['Prediction'] == 1

false_positive_mask = label_mask & pred_mask
false_positives = results_df[false_positive_mask].copy()
shap_values_fp = shap_values_all_class1[false_positive_mask]
X_fp = X_all[false_positive_mask]

print(f"Predicted False Negatives {len(false_positives)}")

plt.figure(figsize=(14, 10))

shap.decision_plot(
    expected_value,
    shap_values_fp,
    features=X_fp,
    feature_names=artifacts['feature_names'],
    show=False,
    feature_display_range=slice(-1, -31, -1),  #show top 30 features
    xlim=[0, 1]
)

plt.title(f'Decision Plot: False Negatives', 
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig("../Figures/SHAP_Analysis/Decision_Plot_False_Negatives.svg", dpi=300, bbox_inches='tight')

plt.show()

In [81]:
plt.figure(figsize=(12, 8))
top_80_importance = mean_abs_shap_all[sorted_indices[:num_features_80]]
sns.barplot(x=top_80_importance, y=top_features_80)
plt.xlabel('Mean Absolute SHAP Value', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.title(f'Top {num_features_80} Most Important Features (80% Cumulative Importance) ChEMBL Screening', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../Figures/SHAP_Analysis/top_features_80_threshold.svg', dpi=300, bbox_inches='tight')
plt.show()

In [80]:
# Bar chart for top features at 90% threshold
plt.figure(figsize=(12, 10))
top_90_importance = mean_abs_shap_all[sorted_indices[:num_features_90]]
sns.barplot(x=top_90_importance, y=top_features_90)
plt.xlabel('Mean Absolute SHAP Value', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.title(f'Top {num_features_90} Most Important Features (90% Cumulative Importance) ChEMBL Screening', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../Figures/SHAP_Analysis/top_features_90_threshold.svg', dpi=300, bbox_inches='tight')
plt.show()